# Notebook 3 — Train / Validation / Test Split

## Purpose

Split the labelled order table **before detailed exploratory analysis** so decisions made during EDA cannot leak information from validation or test data.

## Input artifact

`data/processed/orders_labelled.parquet` from Notebook 2.

## Output artifacts

- `data/processed/train.parquet`
- `data/processed/validation.parquet`
- `data/processed/test.parquet`

## Unit of analysis

One eligible delivered order (`order_id`).

## Split strategy

A deterministic chronological split based on `order_purchase_timestamp`:

- earliest 70% → training;
- next 15% → validation;
- newest 15% → test.

This simulates the real production situation: learn from older orders and predict later orders. A random split would usually preserve the label ratio more closely, but it would mix past and future business periods and could produce an overly optimistic evaluation.

## Dataset roles

- **Train:** fit transformations and models; Notebook 4 performs detailed EDA on this split only.
- **Validation:** compare model choices and tune hyperparameters.
- **Test:** untouched until the final evaluation in Notebook 6.

## 1. Import the required libraries

Only deterministic file, hashing, numerical, and table operations are required. No database connection is needed because this notebook consumes Notebook 2's artifact.

In [1]:
from hashlib import sha256
from pathlib import Path

import numpy as np
import pandas as pd

print(f"NumPy version: {np.__version__}")
print(f"pandas version: {pd.__version__}")
print("Imports completed successfully.")

NumPy version: 2.5.2
pandas version: 3.0.5
Imports completed successfully.


## 2. Resolve project and artifact paths

The notebook can run from either the repository root or the `notebooks` directory. All paths remain relative to the project rather than a particular computer.

In [2]:
WORKING_DIR = Path.cwd().resolve()
PROJECT_ROOT = WORKING_DIR.parent if WORKING_DIR.name == "notebooks" else WORKING_DIR

assert (PROJECT_ROOT / "docker-compose.yml").exists(), (
    "Project root could not be identified. Run this notebook from the repository "
    "root or its notebooks directory."
)

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
INPUT_PATH = PROCESSED_DATA_DIR / "orders_labelled.parquet"

OUTPUT_PATHS = {
    "train": PROCESSED_DATA_DIR / "train.parquet",
    "validation": PROCESSED_DATA_DIR / "validation.parquet",
    "test": PROCESSED_DATA_DIR / "test.parquet",
}

assert INPUT_PATH.exists(), (
    f"Missing input artifact: {INPUT_PATH}. Run Notebook 2 first."
)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Input artifact: {INPUT_PATH}")
for split_name, split_path in OUTPUT_PATHS.items():
    print(f"{split_name.title()} artifact: {split_path}")

Project root: D:\Documents\mlops-olist
Input artifact: D:\Documents\mlops-olist\data\processed\orders_labelled.parquet
Train artifact: D:\Documents\mlops-olist\data\processed\train.parquet
Validation artifact: D:\Documents\mlops-olist\data\processed\validation.parquet
Test artifact: D:\Documents\mlops-olist\data\processed\test.parquet


## 3. Load and validate the labelled table

This verifies the contract from Notebook 2: 96,470 rows, 67 columns, unique order IDs, a complete binary target, and a usable purchase timestamp for every labelled order.

In [3]:
labelled_orders = pd.read_parquet(INPUT_PATH)

required_columns = {
    "order_id",
    "order_purchase_timestamp",
    "order_estimated_delivery_date",
    "order_delivered_customer_date",
    "delivery_delay_days",
    "is_late",
}
missing_required_columns = sorted(required_columns - set(labelled_orders.columns))

labelled_orders["order_purchase_timestamp"] = pd.to_datetime(
    labelled_orders["order_purchase_timestamp"], errors="coerce"
)

print(f"Input shape: {labelled_orders.shape}")
print(f"Unique order IDs: {labelled_orders['order_id'].nunique()}")
print(f"Duplicated order IDs: {labelled_orders['order_id'].duplicated().sum()}")
print(f"Missing purchase timestamps: {labelled_orders['order_purchase_timestamp'].isna().sum()}")
print(f"Missing labels: {labelled_orders['is_late'].isna().sum()}")
print(f"Label values: {sorted(labelled_orders['is_late'].unique())}")
print(f"Missing required columns: {missing_required_columns}")

assert labelled_orders.shape == (96_470, 67), "Unexpected Notebook 2 artifact shape."
assert not missing_required_columns, "Required columns are missing."
assert labelled_orders["order_id"].notna().all()
assert labelled_orders["order_id"].is_unique
assert labelled_orders["order_purchase_timestamp"].notna().all()
assert labelled_orders["is_late"].notna().all()
assert set(labelled_orders["is_late"].unique()) == {0, 1}

Input shape: (96470, 67)
Unique order IDs: 96470
Duplicated order IDs: 0
Missing purchase timestamps: 0
Missing labels: 0
Label values: [np.int8(0), np.int8(1)]
Missing required columns: []


## 4. Perform only the analysis needed to choose the split

We inspect the overall date range, monthly order volume, and monthly late rate. This is split-design analysis—not detailed feature exploration. It tells us whether the dataset spans time and whether the target changes across business periods.

In [4]:
overall_start = labelled_orders["order_purchase_timestamp"].min()
overall_end = labelled_orders["order_purchase_timestamp"].max()

print(f"Overall purchase range: {overall_start} to {overall_end}")
print(f"Overall duration: {(overall_end - overall_start).days} days")
print(f"Overall late rate: {labelled_orders['is_late'].mean():.4%}")

monthly_split_analysis = (
    labelled_orders.assign(
        purchase_month=labelled_orders["order_purchase_timestamp"].dt.to_period("M").astype(str)
    )
    .groupby("purchase_month", as_index=False)
    .agg(
        order_count=("order_id", "size"),
        late_count=("is_late", "sum"),
        late_rate=("is_late", "mean"),
    )
)
monthly_split_analysis["late_rate_percent"] = (
    monthly_split_analysis["late_rate"].mul(100).round(4)
)

display(monthly_split_analysis[[
    "purchase_month", "order_count", "late_count", "late_rate_percent"
]])

Overall purchase range: 2016-09-15 12:16:38 to 2018-08-29 15:00:37
Overall duration: 713 days
Overall late rate: 6.7731%


,purchase_month,order_count,late_count,late_rate_percent
0,2016-09,1,1,100.0000
1,2016-10,265,2,0.7547
2,2016-12,1,0,0.0000
3,2017-01,750,22,2.9333
4,2017-02,1653,49,2.9643
5,2017-03,2546,116,4.5562
6,2017-04,2303,151,6.5567
7,2017-05,3545,106,2.9901
8,2017-06,3135,95,3.0303
9,2017-07,3872,108,2.7893


## 5. Define deterministic split sizes

The fractions sum to 100%. Integer boundaries are calculated once from the total row count. Any rounding remainder goes to the test split, so no row is lost.

In [5]:
TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15
TEST_FRACTION = 0.15

assert np.isclose(
    TRAIN_FRACTION + VALIDATION_FRACTION + TEST_FRACTION,
    1.0,
)

row_count = len(labelled_orders)
train_end = int(row_count * TRAIN_FRACTION)
validation_end = train_end + int(row_count * VALIDATION_FRACTION)

expected_sizes = {
    "train": train_end,
    "validation": validation_end - train_end,
    "test": row_count - validation_end,
}

print(f"Total rows: {row_count}")
for split_name, expected_size in expected_sizes.items():
    print(
        f"Expected {split_name} rows: {expected_size} "
        f"({expected_size / row_count:.4%})"
    )

Total rows: 96470
Expected train rows: 67529 (70.0000%)
Expected validation rows: 14470 (14.9995%)
Expected test rows: 14471 (15.0005%)


## 6. Sort by time and create the three splits

`order_id` is the secondary sort key, making the result deterministic even when two orders share the same timestamp. There is no random shuffle.

In [6]:
chronological_orders = (
    labelled_orders
    .sort_values(["order_purchase_timestamp", "order_id"], kind="mergesort")
    .reset_index(drop=True)
)

train = chronological_orders.iloc[:train_end].copy().reset_index(drop=True)
validation = chronological_orders.iloc[train_end:validation_end].copy().reset_index(drop=True)
test = chronological_orders.iloc[validation_end:].copy().reset_index(drop=True)

splits = {
    "train": train,
    "validation": validation,
    "test": test,
}

for split_name, split_frame in splits.items():
    print(f"{split_name.title()} shape: {split_frame.shape}")
    assert len(split_frame) == expected_sizes[split_name]

Train shape: (67529, 67)
Validation shape: (14470, 67)
Test shape: (14471, 67)


## 7. Audit date ranges and label balance

Because the split is chronological rather than stratified, class ratios are allowed to differ. The differences are important real-world evidence of temporal distribution shift, so they are measured rather than hidden.

In [7]:
overall_late_rate = labelled_orders["is_late"].mean()

split_summary_rows = []
for split_name, split_frame in splits.items():
    split_late_rate = split_frame["is_late"].mean()
    split_summary_rows.append(
        {
            "split": split_name,
            "rows": len(split_frame),
            "percent_of_data": len(split_frame) / len(labelled_orders) * 100,
            "purchase_start": split_frame["order_purchase_timestamp"].min(),
            "purchase_end": split_frame["order_purchase_timestamp"].max(),
            "on_time_or_early": int(split_frame["is_late"].eq(0).sum()),
            "late": int(split_frame["is_late"].eq(1).sum()),
            "late_rate_percent": split_late_rate * 100,
            "difference_from_overall_pp": (split_late_rate - overall_late_rate) * 100,
        }
    )

split_summary = pd.DataFrame(split_summary_rows)
display(
    split_summary.style.format(
        {
            "percent_of_data": "{:.4f}%",
            "late_rate_percent": "{:.4f}%",
            "difference_from_overall_pp": "{:+.4f}",
        }
    )
)

for split_name, split_frame in splits.items():
    assert set(split_frame["is_late"].unique()) == {0, 1}, (
        f"{split_name} does not contain both target classes."
    )

,split,rows,percent_of_data,purchase_start,purchase_end,on_time_or_early,late,late_rate_percent,difference_from_overall_pp
0,train,67529,70.0000%,2016-09-15 12:16:38,2018-04-15 20:12:35,62239,5290,7.8337%,+1.0606
1,validation,14470,14.9995%,2018-04-15 20:17:11,2018-06-21 08:29:29,13846,624,4.3124%,-2.4607
2,test,14471,15.0005%,2018-06-21 08:41:07,2018-08-29 15:00:37,13851,620,4.2844%,-2.4887


## 8. Prove chronological separation

Every training timestamp must be at or before every validation timestamp, and every validation timestamp must be at or before every test timestamp. Equal boundary timestamps are acceptable; the stable `order_id` tie-breaker still assigns each order exactly once.

In [8]:
train_max_time = train["order_purchase_timestamp"].max()
validation_min_time = validation["order_purchase_timestamp"].min()
validation_max_time = validation["order_purchase_timestamp"].max()
test_min_time = test["order_purchase_timestamp"].min()

print(f"Train maximum timestamp: {train_max_time}")
print(f"Validation minimum timestamp: {validation_min_time}")
print(f"Validation maximum timestamp: {validation_max_time}")
print(f"Test minimum timestamp: {test_min_time}")

assert train_max_time <= validation_min_time
assert validation_max_time <= test_min_time
print("Chronological boundary validation passed.")

Train maximum timestamp: 2018-04-15 20:12:35
Validation minimum timestamp: 2018-04-15 20:17:11
Validation maximum timestamp: 2018-06-21 08:29:29
Test minimum timestamp: 2018-06-21 08:41:07
Chronological boundary validation passed.


## 9. Prove there is no overlap or row loss

The order-ID sets must be pairwise disjoint. Recombining all three splits must exactly reconstruct the labelled input table after deterministic sorting.

In [9]:
train_ids = set(train["order_id"])
validation_ids = set(validation["order_id"])
test_ids = set(test["order_id"])

overlap_counts = {
    "train_validation": len(train_ids & validation_ids),
    "train_test": len(train_ids & test_ids),
    "validation_test": len(validation_ids & test_ids),
}

for overlap_name, overlap_count in overlap_counts.items():
    print(f"{overlap_name} overlap: {overlap_count}")
    assert overlap_count == 0

all_split_ids = train_ids | validation_ids | test_ids
source_ids = set(labelled_orders["order_id"])

print(f"Source unique IDs: {len(source_ids)}")
print(f"Combined split IDs: {len(all_split_ids)}")
print(f"Missing IDs after split: {len(source_ids - all_split_ids)}")
print(f"Unexpected IDs after split: {len(all_split_ids - source_ids)}")

assert all_split_ids == source_ids
assert sum(len(frame) for frame in splits.values()) == len(labelled_orders)

reconstructed = (
    pd.concat([train, validation, test], ignore_index=True)
    .sort_values("order_id")
    .reset_index(drop=True)
)
source_sorted = labelled_orders.sort_values("order_id").reset_index(drop=True)

pd.testing.assert_frame_equal(
    reconstructed,
    source_sorted,
    check_dtype=True,
    check_like=False,
)

print("Exact reconstruction validation passed.")

train_validation overlap: 0
train_test overlap: 0
validation_test overlap: 0


Source unique IDs: 96470
Combined split IDs: 96470
Missing IDs after split: 0
Unexpected IDs after split: 0


Exact reconstruction validation passed.


## 10. Record columns that must not become model inputs

The split artifacts intentionally preserve the labelled table. Later, Notebook 5 will build the prediction feature table and remove outcome-derived, post-delivery, identifier, and target columns. Listing obvious leakage now makes that contract explicit.

In [10]:
known_non_feature_columns = [
    "is_late",
    "delivery_delay_days",
    "order_delivered_customer_date",
    "order_delivered_carrier_date",
    "post_delivery_review_count",
    "post_delivery_review_score_mean",
    "post_delivery_review_score_min",
    "post_delivery_review_score_max",
    "post_delivery_review_title_count",
    "post_delivery_review_message_count",
    "post_delivery_review_message_length_mean",
    "post_delivery_review_creation_date_min",
    "post_delivery_review_answer_timestamp_max",
]

present_non_feature_columns = [
    column for column in known_non_feature_columns if column in labelled_orders.columns
]

print("Known target/outcome/post-delivery columns preserved for now:")
for column in present_non_feature_columns:
    print(f"- {column}")

print(
    "These columns are preserved in the split artifacts for auditability, "
    "but they are not approved model features."
)

Known target/outcome/post-delivery columns preserved for now:
- is_late
- delivery_delay_days
- order_delivered_customer_date
- order_delivered_carrier_date
- post_delivery_review_count
- post_delivery_review_score_mean
- post_delivery_review_score_min
- post_delivery_review_score_max
- post_delivery_review_title_count
- post_delivery_review_message_count
- post_delivery_review_message_length_mean
- post_delivery_review_creation_date_min
- post_delivery_review_answer_timestamp_max
These columns are preserved in the split artifacts for auditability, but they are not approved model features.


## 11. Save, reload, and hash all split artifacts

Each split is written separately and reloaded immediately. Exact dataframe equality verifies the artifact contract; SHA-256 identifies the precise local file bytes.

In [11]:
artifact_records = []

for split_name, split_frame in splits.items():
    split_path = OUTPUT_PATHS[split_name]
    split_frame.to_parquet(split_path, index=False)
    reloaded_split = pd.read_parquet(split_path)

    pd.testing.assert_frame_equal(
        split_frame,
        reloaded_split,
        check_dtype=True,
        check_like=False,
    )

    artifact_records.append(
        {
            "split": split_name,
            "path": str(split_path),
            "rows": len(reloaded_split),
            "columns": len(reloaded_split.columns),
            "size_bytes": split_path.stat().st_size,
            "sha256": sha256(split_path.read_bytes()).hexdigest().upper(),
        }
    )

artifact_summary = pd.DataFrame(artifact_records)
display(artifact_summary)
print("All Parquet round-trip validations passed.")

,split,path,rows,columns,size_bytes,sha256
0,train,D:\Documents\mlops-olist\data\processed\train....,67529,67,14908154,7E84C17DB5D2262E4BAD09326E7784007CFD0F79C74698...
1,validation,D:\Documents\mlops-olist\data\processed\valida...,14470,67,3440723,C01444C587B6BD58AF45993F4DEE49D711F410D04CBC80...
2,test,D:\Documents\mlops-olist\data\processed\test.p...,14471,67,3439610,DEB4EE2A6E8398A8F4914D4FC3CFBAA2EF3F6C7D0DF3FF...


All Parquet round-trip validations passed.


## 12. Completion summary and handoff

Notebook 3 is complete when:

- the labelled data is split chronologically and deterministically;
- train, validation, and test contain both target classes;
- their date ranges are chronologically ordered;
- no order appears in more than one split;
- no order is lost or added;
- all three saved artifacts pass exact reload validation.

Notebook 4 will read **only `train.parquet`** for detailed EDA. Validation and test remain closed while we study distributions, missingness, unusual values, feature relationships, geography, dates, and possible leakage.

In [12]:
print("NOTEBOOK 3 FINAL RESULT")
print("Split strategy: chronological 70% / 15% / 15%")
print(f"Input rows: {len(labelled_orders)}")

for split_name, split_frame in splits.items():
    split_start = split_frame["order_purchase_timestamp"].min()
    split_end = split_frame["order_purchase_timestamp"].max()
    split_late_rate = split_frame["is_late"].mean()
    split_hash = artifact_summary.loc[
        artifact_summary["split"].eq(split_name), "sha256"
    ].iloc[0]

    print(
        f"{split_name.title()}: rows={len(split_frame)}, "
        f"late_rate={split_late_rate:.4%}, "
        f"dates={split_start} to {split_end}"
    )
    print(f"{split_name.title()} SHA-256: {split_hash}")

print(f"Cross-split overlaps: {sum(overlap_counts.values())}")
print(f"Missing or unexpected IDs: {len(source_ids ^ all_split_ids)}")
print("Detailed EDA allowed on: TRAIN ONLY")
print("Ready for Notebook 4: YES")

NOTEBOOK 3 FINAL RESULT
Split strategy: chronological 70% / 15% / 15%
Input rows: 96470
Train: rows=67529, late_rate=7.8337%, dates=2016-09-15 12:16:38 to 2018-04-15 20:12:35
Train SHA-256: 7E84C17DB5D2262E4BAD09326E7784007CFD0F79C7469831B457C390EE971A9C
Validation: rows=14470, late_rate=4.3124%, dates=2018-04-15 20:17:11 to 2018-06-21 08:29:29
Validation SHA-256: C01444C587B6BD58AF45993F4DEE49D711F410D04CBC80C24112BD94CBE4C5A0
Test: rows=14471, late_rate=4.2844%, dates=2018-06-21 08:41:07 to 2018-08-29 15:00:37
Test SHA-256: DEB4EE2A6E8398A8F4914D4FC3CFBAA2EF3F6C7D0DF3FFB755C4B981457D76D2
Cross-split overlaps: 0
Missing or unexpected IDs: 0
Detailed EDA allowed on: TRAIN ONLY
Ready for Notebook 4: YES
